In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/data'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# GP-Based Soay Sheep IBM

Fit GP vital rates to the 1,000-row baseline sample, generate an IBM dataset, and construct the fitted GP IPM. Recruitment depends on mother's age and recruit size depends on mother's size. Offspring sex probability is fixed at 0.5.

In [ ]:
from pathlib import Path; import sys; sys.path.insert(0, str(Path.cwd().parent.parent))
import numpy as np
import pandas as pd
import gpflow
import tensorflow as tf
from experiments.soay.common.ipm_utils import calculate_ipm_metrics, make_gp_ipm_agesize, sigmoid

In [ ]:
project_dir = PROJECT_ROOT
print(project_dir)

In [ ]:
baseline = pd.read_csv(GENERATED / 'soay/age_size/data/baseline_ibm_data_agesize_1000.csv')
growth = baseline[["z", 'age', "z1"]].dropna()
survival = baseline[["z", 'age', "Surv"]].dropna()
reproduction = baseline.loc[baseline["age"] >= 1,["z", "age", "Repr"],].dropna()
recruitment = baseline[["Recr", 'age']].dropna()
recruit_size = baseline[["z", "Rcsz"]].dropna()

# Growth is age-size dependent
X_growth = growth[["z", 'age']].to_numpy(np.float64)
Y_growth = growth[["z1"]].to_numpy(np.float64)

# Survival is age-size dependent
X_surv = survival[["z", 'age']].to_numpy(np.float64)
Y_surv = survival[["Surv"]].to_numpy(np.float64)

# Reproduction is age-size dependent
X_repr = reproduction[["z", 'age']].to_numpy(np.float64)
Y_repr = reproduction[["Repr"]].to_numpy(np.float64)

# Recruitment only depends on mother's age
X_recr = recruitment[['age']].to_numpy(np.float64)
Y_recr = recruitment[['Recr']].to_numpy(np.float64)

# Recruit size only depends on mother's size
X_rcsz = recruit_size[["z"]].to_numpy(np.float64)
Y_rcsz = recruit_size[["Rcsz"]].to_numpy(np.float64)

In [ ]:
model_growth = gpflow.models.GPR((X_growth, Y_growth), gpflow.kernels.SquaredExponential(lengthscales=[1.0, 1.0]))
model_surv = gpflow.models.VGP((X_surv, Y_surv), gpflow.kernels.SquaredExponential(lengthscales=[1.0, 1.0]), gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
model_repr = gpflow.models.VGP((X_repr, Y_repr), gpflow.kernels.SquaredExponential(lengthscales=[1.0, 1.0]), gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
model_recr = gpflow.models.VGP((X_recr, Y_recr), gpflow.kernels.SquaredExponential(), gpflow.likelihoods.Bernoulli(invlink=tf.sigmoid))
model_rcsz = gpflow.models.GPR((X_rcsz, Y_rcsz), gpflow.kernels.SquaredExponential())
optimizer = gpflow.optimizers.Scipy()
for model in [model_growth, model_surv, model_repr, model_recr,  model_rcsz]:
    optimizer.minimize(model.training_loss, model.trainable_variables)

for name, model in {"Growth": model_growth, 
                    "Survival": model_surv, 
                    "Reproduction": model_repr, 
                    'Recruitment': model_recr,
                    "Recruit Size": model_rcsz}.items(): 
    print(f"\n{name}"); gpflow.utilities.print_summary(model)


params = {"growth_noise": float(np.sqrt(model_growth.likelihood.variance.numpy())), 
          "rcsz_noise": float(np.sqrt(model_rcsz.likelihood.variance.numpy())), 
          "female_prob": 0.5}

funcs = {"growth": model_growth, 
         "surv": model_surv, 
         "repr": model_repr,
         'recr': model_recr, 
         "rcsz": model_rcsz}

In [ ]:
def sample_gp(model, X):
    X = tf.convert_to_tensor(np.asarray(X, dtype=np.float64), dtype=gpflow.default_float())
    return model.predict_f_samples(X, num_samples=1).numpy().ravel()

def predict_gp_mean(model, X):
    X = tf.convert_to_tensor(np.asarray(X, dtype=np.float64), dtype=gpflow.default_float())
    mean, _ = model.predict_f(X)
    return mean.numpy().ravel()

In [ ]:
rng = np.random.default_rng(562)
tf.random.set_seed(562)
initial_pop_size = 100
max_age = 7
min_reproductive_age = 1
initial_mean = predict_gp_mean(model_rcsz, np.array([[3.2]]))[0]
z = rng.normal(initial_mean, params["rcsz_noise"], initial_pop_size)
age = np.zeros(initial_pop_size, dtype=int)

frames = []
year = 1

while year <= 10 and 0 < len(z) < 5000:
    n = len(z)
    print('Current Population Size:', n)
    age_size_inputs = np.column_stack([z, age])

    survival_probability = np.clip(
        sigmoid(sample_gp(model_surv, age_size_inputs)),
        0,
        1,
    )
    surv = rng.binomial(1, survival_probability)
    survived = surv == 1

    z1 = np.full(n, np.nan)
    if survived.any():
        growth_inputs = np.column_stack([z[survived], age[survived]])
        growth_mean = sample_gp(model_growth, growth_inputs)
        z1[survived] = rng.normal(
            growth_mean,
            params["growth_noise"],
        )

    repr_ = np.full(n, np.nan)
    repr_[survived] = 0

    eligible_to_reproduce = survived & (age >= min_reproductive_age)

    if eligible_to_reproduce.any():
        reproduction_inputs = np.column_stack([
            z[eligible_to_reproduce],
            age[eligible_to_reproduce],
        ])
        reproduction_probability = np.clip(
            sigmoid(sample_gp(model_repr, reproduction_inputs)),
            0,
            1,
        )
        repr_[eligible_to_reproduce] = rng.binomial(
            1,
            reproduction_probability,
        )

    reproduced = eligible_to_reproduce & (repr_ == 1)

    sex = np.full(n, np.nan)
    sex[reproduced] = rng.binomial(
        1,
        params.get("female_prob", 0.5),
        reproduced.sum(),
    )
    female = reproduced & (sex == 1)

    recr = np.full(n, np.nan)
    if female.any():
        recruitment_inputs = age[female].reshape(-1, 1)
        recruitment_probability = np.clip(
            sigmoid(sample_gp(model_recr, recruitment_inputs)),
            0,
            1,
        )
        recr[female] = rng.binomial(
            1,
            recruitment_probability,
        )

    recruited = female & (recr == 1)

    rcsz = np.full(n, np.nan)
    if recruited.any():
        recruit_size_inputs = z[recruited].reshape(-1, 1)
        recruit_size_mean = sample_gp(
            model_rcsz,
            recruit_size_inputs,
        )
        rcsz[recruited] = rng.normal(
            recruit_size_mean,
            params["rcsz_noise"],
        )

    frames.append(
        pd.DataFrame({
            "z": z,
            "age": age,
            "Surv": surv,
            "z1": z1,
            "Repr": repr_,
            "Sex": sex,
            "Recr": recr,
            "Rcsz": rcsz,
            "yr": year,
        })
    )

    z = np.concatenate([rcsz[recruited], z1[survived]])
    age = np.concatenate([
        np.zeros(recruited.sum(), dtype=int),
        np.minimum(age[survived] + 1, max_age + 1),
    ])
    year += 1

sim_data = pd.concat(frames, ignore_index=True)

if len(sim_data) > 1000:
    sim_data = sim_data.sample(
        1000,
        replace=False,
        random_state=562,
    ).reset_index(drop=True)

In [ ]:
sim_data.describe().round(3)

In [ ]:
sim_data.to_csv("gp_ibm_dataset_agesize.csv", index=False)

In [ ]:
L, U, n_mesh = 2, 3.5, 250
gp_ipm = make_gp_ipm_agesize(
    n_mesh,
    L,
    U,
    max_age,
    params,
    funcs,
    correction=True,
    min_reproductive_age=min_reproductive_age,
)
mesh = gp_ipm["mesh_points"]

In [ ]:
gp_metrics = calculate_ipm_metrics(gp_ipm["K"], gp_ipm["reproduction_probability"])

In [ ]:
gp_metrics['lambda']

In [ ]:
np.savez_compressed("true_gp_metrics.npz", **gp_metrics)
print(f"GP IBM Lambda: {gp_metrics['lambda']:.6f}")